# 04. Refined Risk Segments (Semantic Version)

本 Notebook 實作了進階的風險分群邏輯，解決了傳統決策樹數值編碼不具語義的問題。 

### 核心優化點：
1. **全變數對齊 (11 Variables)**：對齊模型訓練時使用的所有 11 個特徵。
2. **語義化編碼 (Semantic Encoding)**：將 `Sex`, `Smoker`, `Insurance_Plan` 與 `Preferred_Class` 轉為清晰的名詞標籤。
3. **規則優化器 (Rule Optimizer)**：合併冗餘的數值區間，生成如 `51.5 < Age <= 60.5` 的簡潔規則。
4. **資料品質治理**：針對 `Preferred_Class` 的混亂標籤 (`1.0` vs `1`, `NA` vs `U`) 進行自動正規化。

In [8]:
import pandas as pd
import numpy as np
import json
import os
import re
import lightgbm as lgb
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import LabelEncoder

# Paths
DATA_PATH = '../../data/ilec_cleaned.parquet'
MODEL_PATH = '../../models/lgbm_mortality_offset_poisson.txt'
OUTPUT_DIR = '../../knowledge_base/segments'
SPOTLIGHT_DIR = f'{OUTPUT_DIR}/spotlight'

os.makedirs(SPOTLIGHT_DIR, exist_ok=True)
print("Setup complete.")

Setup complete.


## 1. 規則最佳化與翻譯引擎

這部分函數負責將決策樹內部的數字門檻轉換為人類可讀的精算語意。

In [9]:
FACE_BAND_MAP = {
    1: '1 - 9,999', 2: '10,000 - 24,999', 3: '25,000 - 49,999', 4: '50,000 - 99,999', 
    5: '100,000 - 249,999', 6: '250,000 - 499,999', 7: '500,000 - 999,999', 
    8: '1,000,000 - 2,499,999', 9: '2,500,000 - 4,999,999', 10: '5,000,000 - 9,999,999', 11: '10,000,000+'
}

def translate_rule(feat, low, high):
    is_true = (low > -float('inf') and low >= 0.4)
    is_false = (high < float('inf') and high <= 0.6)

    if feat == 'Is_Male':
        if is_true: return "Sex is Male"
        if is_false: return "Sex is Female"
    if feat == 'Is_Smoker_Known':
        if is_true: return "Smoker Status is Known"
        if is_false: return "Smoker Status is Unknown"
    if feat == 'Is_Smoker':
        if is_true: return "Is Smoker"
        if is_false: return "Non-Smoker"
    if feat.startswith('Plan_'):
        plan = feat.replace('Plan_', '')
        if is_true: return f"Insurance Plan is {plan}"
        if is_false: return f"Insurance Plan is NOT {plan}"
    if feat.startswith('Pref_'):
        pref = feat.replace('Pref_', '')
        if is_true: return f"Preferred Class is {pref}"
        if is_false: return f"Preferred Class is NOT {pref}"
    if feat == 'Face_Amount_Band_Val':
        if low != -float('inf') and high != float('inf'):
            l_idx, h_idx = int(np.ceil(low)), int(np.floor(high))
            return f"Face Amount Band: {FACE_BAND_MAP.get(l_idx, l_idx)} to {FACE_BAND_MAP.get(h_idx, h_idx)}"
        elif low != -float('inf'):
            l_idx = int(np.ceil(low))
            return f"Face Amount Band > {FACE_BAND_MAP.get(l_idx, l_idx)}"
        else:
            h_idx = int(np.floor(high))
            return f"Face Amount Band <= {FACE_BAND_MAP.get(h_idx, h_idx)}"

    # Numeric
    if low != -float('inf') and high != float('inf'):
        return f"{low:.1f} < {feat} <= {high:.1f}"
    elif low != -float('inf'):
        return f"{feat} > {low:.1f}"
    else:
        return f"{feat} <= {high:.1f}"

def get_leaf_path(tree, leaf_id, feature_names):
    tree_ = tree.tree_
    def find_path(node, target, path=[]):
        if node == target: return path
        l, r = tree_.children_left[node], tree_.children_right[node]
        if l != -1:
            res = find_path(l, target, path + [(node, 'left')])
            if res: return res
        if r != -1:
            res = find_path(r, target, path + [(node, 'right')])
            if res: return res
        return None
    
    path = find_path(0, leaf_id)
    bounds = {}
    for node, direction in path:
        feat = feature_names[tree_.feature[node]]
        thresh = tree_.threshold[node]
        if feat not in bounds: bounds[feat] = {'min': -float('inf'), 'max': float('inf')}
        if direction == 'right': bounds[feat]['min'] = max(bounds[feat]['min'], thresh)
        else: bounds[feat]['max'] = min(bounds[feat]['max'], thresh)
    
    rules = []
    for feat, b in bounds.items():
        rule = translate_rule(feat, b['min'], b['max'])
        if rule: rules.append(rule)
    return rules

## 2. 資料載入與特徵工程

In [10]:
print("Loading data...")
df = pd.read_parquet(DATA_PATH)
df = df.sample(frac=0.1, random_state=42).reset_index(drop=True)

print("Normalizing Preferred Class & Smoking Status...")
df['Preferred_Class_Clean'] = df['Preferred_Class'].astype(str).str.strip().str.upper().replace({'NA':'U','NAN':'U','NONE':'U','UNKNOWN':'U','N/A':'U'}).str.replace('.0', '', regex=False)
df['Is_Male'] = (df['Sex'] == 'M').astype(int)
df['Is_Smoker_Known'] = (df['Smoker_Status'] != 'U').astype(int)
df['Is_Smoker'] = (df['Smoker_Status'] == 'S').astype(int)
df['Face_Amount_Band_Val'] = df['Face_Amount_Band'].str.extract('(\d+)').astype(float)

print("Generating Dummies...")
plan_dummies = pd.get_dummies(df['Insurance_Plan'], prefix='Plan').astype(int)
pref_dummies = pd.get_dummies(df['Preferred_Class_Clean'], prefix='Pref').astype(int)
soa_dummies = pd.concat([
    pd.get_dummies(df['SOA_Post_Lvl_Ind'], prefix='SOA_Post'),
    pd.get_dummies(df['SOA_Antp_Lvl_TP'], prefix='SOA_Antp'),
    pd.get_dummies(df['SOA_Guar_Lvl_TP'], prefix='SOA_Guar')
], axis=1).astype(int)

df = pd.concat([df, plan_dummies, pref_dummies, soa_dummies], axis=1)
print(f"Feature Engineering complete. Current columns: {len(df.columns)}")

Loading data...


<>:10: SyntaxWarning: invalid escape sequence '\d'
<>:10: SyntaxWarning: invalid escape sequence '\d'
/var/folders/2q/db42cqnn7nx7pvw9p0lf57hm0000gp/T/ipykernel_37281/1419346782.py:10: SyntaxWarning: invalid escape sequence '\d'
  df['Face_Amount_Band_Val'] = df['Face_Amount_Band'].str.extract('(\d+)').astype(float)


Normalizing Preferred Class & Smoking Status...
Generating Dummies...
Feature Engineering complete. Current columns: 56


## 3. 模型校準與基準計算

In [11]:
print("Loading LightGBM model...")
model = lgb.Booster(model_file=MODEL_PATH)

FEATURES_LGBM = ['Attained_Age', 'Issue_Age', 'Duration', 'Sex', 'Smoker_Status', 
                 'Insurance_Plan', 'Face_Amount_Band', 'Preferred_Class', 
                 'SOA_Post_Lvl_Ind', 'SOA_Antp_Lvl_TP', 'SOA_Guar_Lvl_TP']

X_lgbm = df[FEATURES_LGBM].copy()
for col in X_lgbm.select_dtypes(['object', 'category']).columns:
    X_lgbm[col] = LabelEncoder().fit_transform(X_lgbm[col].astype(str))

df['Predicted_Rate'] = model.predict(X_lgbm.values)
df['Expected_Deaths'] = df['Predicted_Rate'] * df['Policies_Exposed']
df['Actual_Rate'] = df['Death_Count'] / df['Policies_Exposed'].replace(0, np.nan)
overall_rate = df['Death_Count'].sum() / df['Policies_Exposed'].sum()
print(f"Overall Death Rate: {overall_rate:.6f}")

Loading LightGBM model...
Overall Death Rate: 0.009666


## 4. 生成分層分群

In [12]:
# 定義變數集合
COV_FEAT = ['Attained_Age', 'Duration', 'Is_Male', 'Is_Smoker_Known', 'Is_Smoker'] + plan_dummies.columns.tolist()
SPOT_FEAT = COV_FEAT + ['Issue_Age', 'Face_Amount_Band_Val'] + pref_dummies.columns.tolist() + soa_dummies.columns.tolist()

print("Generating Layer A (Coverage)...")
X_cov, y_cov = df[COV_FEAT], (df['Death_Count'] / df['Expected_Deaths'].replace(0, np.nan)).fillna(1.0).clip(0.1, 10)
cov_tree = DecisionTreeRegressor(max_depth=5, min_samples_leaf=10000, random_state=42)
cov_tree.fit(X_cov, y_cov, sample_weight=df['Policies_Exposed'])

print("Generating Layer B (Spotlight)...")
X_spot, y_spot = df[SPOT_FEAT], (df['Actual_Rate'] / overall_rate).fillna(1.0).clip(0.1, 10)
spot_tree = DecisionTreeRegressor(max_depth=7, min_samples_leaf=5000, random_state=42)
spot_tree.fit(X_spot, y_spot, sample_weight=df['Policies_Exposed'])

print("Training complete.")

Generating Layer A (Coverage)...
Generating Layer B (Spotlight)...
Training complete.


## 5. 匯出 RAG 知識文件

In [13]:
# --- 匯出 Coverage ---
df['Coverage_Leaf'] = cov_tree.apply(X_cov)
cov_agg = df.groupby('Coverage_Leaf').agg({'Policies_Exposed': 'sum', 'Death_Count': 'sum', 'Expected_Deaths': 'sum'}).reset_index()
cov_registry = {"layer": "A", "overall_rate": round(overall_rate, 6), "segments": []}

for _, row in cov_agg.iterrows():
    leaf_id = int(row['Coverage_Leaf'])
    rules = get_leaf_path(cov_tree, leaf_id, COV_FEAT)
    cov_registry["segments"].append({
        "segment_id": f"COV_{leaf_id:03d}",
        "rule_list": rules,
        "statistics": {
            "exposure": int(row['Policies_Exposed']),
            "ae_ratio": round(row['Death_Count'] / row['Expected_Deaths'], 3),
            "relative_risk": round((row['Death_Count'] / row['Policies_Exposed']) / overall_rate, 3)
        }
    })

with open(os.path.join(OUTPUT_DIR, 'coverage_registry.json'), 'w') as f: json.dump(cov_registry, f, indent=2)

# --- 匯出 Spotlight ---
for f in os.listdir(SPOTLIGHT_DIR): os.remove(os.path.join(SPOTLIGHT_DIR, f))
df['Spotlight_Leaf'] = spot_tree.apply(X_spot)
spot_agg = df.groupby('Spotlight_Leaf').agg({'Policies_Exposed': 'sum', 'Death_Count': 'sum', 'Expected_Deaths': 'sum'}).reset_index()
spot_agg['relative_risk'] = (spot_agg['Death_Count'] / spot_agg['Policies_Exposed']) / overall_rate
anomalous = spot_agg[abs(spot_agg['relative_risk'] - 1) > 0.15].copy()

spotlight_summary = {"layer": "B", "overall_rate": round(overall_rate, 6), "anomalies": []}
for _, row in anomalous.iterrows():
    leaf_id = int(row['Spotlight_Leaf'])
    rules = get_leaf_path(spot_tree, leaf_id, SPOT_FEAT)
    detail = {
        "segment_id": f"SPOT_{leaf_id:03d}", 
        "rule_list": rules,
        "statistics": {
            "exposure": int(row['Policies_Exposed']), 
            "ae_ratio": round(row['Death_Count'] / (row['Expected_Deaths'] + 1e-10), 3),
            "relative_risk": round(row['relative_risk'], 3)
        }
    }
    with open(os.path.join(SPOTLIGHT_DIR, f"SPOT_{leaf_id:03d}.json"), 'w') as f: json.dump(detail, f, indent=2)
    spotlight_summary["anomalies"].append({"segment_id": f"SPOT_{leaf_id:03d}", "rule_list": rules, "ae_ratio": detail["statistics"]["ae_ratio"], "relative_risk": detail["statistics"]["relative_risk"]})

with open(os.path.join(OUTPUT_DIR, 'spotlight_summary.json'), 'w') as f: json.dump(spotlight_summary, f, indent=2)
print("Success: JSON knowledge base updated.")

Success: JSON knowledge base updated.
